## Seminar 6. BERT: zero-shot & training

Well, yet again back to the `ag_news` dataset. One more way to classify news.

In [ ]:
# Environment for week06: BERT: zero-shot, features, fine-tuning.
# Colab or a fresh virtualenv: run this cell once. Locally: `uv sync --group week06` in the repo root instead.
%pip install -q "numpy>=1.26" "pandas>=2.2" "matplotlib>=3.9" "tqdm>=4.66" "torch>=2.4,<3" "transformers>=4.56,<5" "datasets>=3.6,<4" "evaluate>=0.4.3" "accelerate>=1.0" "scikit-learn>=1.5"

In [10]:
from datasets import load_dataset

dataset = load_dataset("fancyzhx/ag_news")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [11]:
from transformers import BertTokenizerFast

tokenizer_fast = BertTokenizerFast.from_pretrained("bert-base-uncased")

In [ ]:
import os
from functools import partial

def tokenize(tokenizer, sample):
    return tokenizer(sample['text'])

dataset = dataset.map(
    partial(tokenize, tokenizer_fast),
    batched=True,
    num_proc=min(8, os.cpu_count() or 1),
    desc="Tokenization",
    batch_size=1000,
)

In [16]:
dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
dataset['test'][0]

{'label': tensor(2),
 'input_ids': tensor([  101, 10069,  2005,  1056,  1050, 11550,  2044,  7566,  9209,  5052,
          3667,  2012,  6769,  2047,  8095,  2360,  2027,  2024,  1005,  9364,
          1005,  2044,  7566,  2007, 16654,  6687,  3813,  2976,  9587, 24848,
          1012,   102]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1])}

## Transformers in Hugging Face

Any model along with its weights can be loaded in the same way as the tokenizer: via `AutoModel` or by importing the specific class.

In [17]:
from transformers import AutoModel

bert_model = AutoModel.from_pretrained('bert-base-uncased')
bert_model

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

Models like BERT are often fine-tuned for smaller tasks such as text classification, token classification, and so on. The output format varies from task to task, and therefore a specific head is required. In `transformers`, all of this is already provided — there is a separate modification of the transformer model for each task. When loading such a model, among other parameters, you can specify the number of classes for prediction.

__Important:__ When loading the model, the weights for the head are not loaded (where would they come from?). Therefore, such a model cannot be used without fine-tuning.

In [18]:
from transformers import AutoModelForSequenceClassification, AutoModelForTokenClassification, AutoModelForMaskedLM

In [19]:
bert_model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=10)
bert_model

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

## Classification Task

During the whole module we've been trying to classify news from the AG-news dataset. As you remember, we've succeeded on our first try with TF-iDF -- accuracy==0.92, and still struggling to outdo this result with the most fancy models.  Now it's BERT's turn.

In [21]:
import numpy as np
import torch
from tqdm.auto import tqdm
import evaluate

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

Today we'll use the accuracy metrics from `evaluate`.

In [22]:
accuracy_metric = evaluate.load('accuracy')

### Zero-shot

During training, BERT learns to unmask hidden tokens so that they fit the context semantically. This means that if we construct an example where the masked token corresponds to a class label in a classification task, BERT will be able to predict it. This usage paradigm is called **zero-shot**, because the model has not been trained on any examples of that specific classification task.

Let’s load the BERT weights for the MLM task. They are available because BERT was originally pre-trained on this exact objective.


In [23]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_model = AutoModelForMaskedLM.from_pretrained("bert-base-uncased")

BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'c

Let's see what BERT can do in a zero-shot mode with the simplest example:

In [24]:
story_desc = "Elephants are playing in the zoo with monkeys."
prompt = "The story is about [MASK]."

In [25]:
tokenized = tokenizer(f'{story_desc} {prompt}', return_tensors='pt')
tokenized

{'input_ids': tensor([[  101, 16825,  2024,  2652,  1999,  1996,  9201,  2007, 17059,  1012,
          1996,  2466,  2003,  2055,   103,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [26]:
output = bert_model(**tokenized)
print(output.logits.shape)

# -3 соответствует токену маски (103)
probs = torch.softmax(output.logits[0, -3], dim=0)
probs, tokens = probs.topk(10)  # Берем 10 наиболее вероятных предсказаний

torch.Size([1, 17, 30522])


In [27]:
for prob, word in zip(probs, tokenizer.convert_ids_to_tokens(tokens)):
    print(f'{word} \tProb: {round(prob.item(), 3)}')

elephants 	Prob: 0.216
monkeys 	Prob: 0.109
animals 	Prob: 0.086
elephant 	Prob: 0.039
children 	Prob: 0.022
monkey 	Prob: 0.02
birds 	Prob: 0.013
lions 	Prob: 0.013
love 	Prob: 0.01
butterflies 	Prob: 0.009


Seems like we're on a right path.

### `Pipeline`

To apply a model, we always need to tokenize the text first. To avoid doing this separately each time, we can use a `pipeline`, where tokenization and model inference are combined into a single step. The first parameter specifies the type of task, and based on it, the appropriate model type is selected (if available).


In [28]:
from transformers import pipeline

pipe = pipeline("fill-mask", model="bert-base-uncased", device=device)

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [29]:
pipe.tokenizer

BertTokenizerFast(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [30]:
pipe.model

BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwi

Let's check, that the result of `pipeline` mathces the result of our previous test:

In [31]:
output = pipe(f'{story_desc} {prompt}', top_k=10)
for out in output:
    print(f'{out["token_str"]} \tProb: {round(out["score"], 3)}')

elephants 	Prob: 0.216
monkeys 	Prob: 0.109
animals 	Prob: 0.086
elephant 	Prob: 0.039
children 	Prob: 0.022
monkey 	Prob: 0.02
birds 	Prob: 0.013
lions 	Prob: 0.013
love 	Prob: 0.01
butterflies 	Prob: 0.009


### News classification

In any classification task, there is usually a fixed set of predefined classes with specific names. For example, in news classification these might be *["World", "Sports", "Business", "Sci/Tech"]*.

We can pass the list of class names to `targets` so that the `pipeline` returns probabilities only for these classes.

In [32]:
label2name = ["world", "sports", "business", "technology"]
name2label = {v: i for i, v in enumerate(label2name)}

In [33]:
prompt = 'This news is about [MASK].'

prompted_dataset = dataset['test'].map(
    lambda sample: {'text': [f'{text} {prompt}' for text in sample['text']]},
    batched=True,
)
prompted_dataset.set_format(type='torch', columns=['text', 'label'])

In [34]:
def zero_shot_predictions(dataset):
    bs = 32
    predictions = []
    for i in tqdm(range(0, len(dataset), bs)):
        output = pipe(dataset[i:i+bs]['text'], targets=label2name, top_k=1)
        predictions.extend([name2label[out[0]['token_str']] for out in output])

    predictions = np.array(predictions)
    return predictions

In [35]:
predictions = zero_shot_predictions(prompted_dataset)

  0%|          | 0/238 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [36]:
accuracy = accuracy_metric.compute(predictions=predictions, references=prompted_dataset['label'])
accuracy['accuracy']

0.4561842105263158

Hm? Something is wrong. Let’s look at it in more detail

In [37]:
per_class_acc = {}
for label in np.unique(prompted_dataset['label']):
    mask = torch.tensor(prompted_dataset['label']) == label
    per_class_acc[label2name[label]] = accuracy_metric.compute(
        predictions=predictions[mask],
        references=torch.tensor(prompted_dataset['label'])[mask]
    )['accuracy']

<ipython-input-37-8159c987a21a>:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask = torch.tensor(prompted_dataset['label']) == label
<ipython-input-37-8159c987a21a>:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  references=torch.tensor(prompted_dataset['label'])[mask]
<ipython-input-37-8159c987a21a>:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask = torch.tensor(prompted_dataset['label']) == label
<ipython-input-37-8159c987a21a>:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detac

In [38]:
per_class_acc

{'world': 0.031578947368421054,
 'sports': 0.5131578947368421,
 'business': 0.9636842105263158,
 'technology': 0.3163157894736842}

For some classes the accuracy is pretty high, while for the others not so great. Obviousy, the names we gave to the classes don't really match with the topic of their examples. Now, let's try to find a more suitable name for the `world` class news. What are the most popular options for the news of this class?

In [39]:
world_prompted_dataset = prompted_dataset.filter(
    lambda sample: sample['label'] == name2label['world'],
)
world_prompted_dataset

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1900
})

In [40]:
from collections import Counter

In [41]:
bs = 32
top100_words = Counter()
for i in tqdm(range(0, len(world_prompted_dataset), bs)):
    with torch.inference_mode():
        output = pipe(world_prompted_dataset[i:i+bs]['text'], top_k=100)

    for out in output:
        top100_words += Counter([out[j]['token_str'] for j in range(len(out))])

  0%|          | 0/60 [00:00<?, ?it/s]

In [42]:
top100_words.most_common(20)

[('us', 1896),
 ('it', 1861),
 ('america', 1837),
 ('me', 1821),
 ('you', 1818),
 ('politics', 1726),
 ('news', 1721),
 ('time', 1687),
 ('him', 1651),
 ('china', 1642),
 ('them', 1596),
 ('history', 1474),
 ('canada', 1427),
 ('democracy', 1425),
 ('war', 1408),
 ('life', 1373),
 ('russia', 1348),
 ('freedom', 1343),
 ('terrorism', 1333),
 ('iran', 1328)]

It seems that `politics` is the most suitable of all of them

In [43]:
label2name = ["politics", "sports", "business", "technology"]
name2label = {v: i for i, v in enumerate(label2name)}

In [44]:
prompt = 'This news is about [MASK].'

prompted_dataset = dataset['test'].map(
    lambda sample: {'text': [f'{text} {prompt}' for text in sample['text']]},
    batched=True,
)
prompted_dataset.set_format(type='torch', columns=['text', 'label'])

In [46]:
predictions = zero_shot_predictions(prompted_dataset)

  0%|          | 0/238 [00:00<?, ?it/s]

In [47]:
accuracy = accuracy_metric.compute(predictions=predictions, references=prompted_dataset['label'])
accuracy['accuracy']

0.6128947368421053

Now that’s a different story.

This example demonstrates that BERT is indeed capable of zero-shot learning. Of course, using it in this raw form is rather pointless — the quality will never be high. It’s much better to fine-tune BERT on the classification task or at least use its outputs as features for a linear model.


## Linear model on top of BERT outputs

BERT was trained on the next sentence prediction task, so the [CLS] token contains information about the entire text. We will use it as a feature vector for a linear model. To extract the outputs of the [CLS] token, we’ll load the corresponding modification of the model.

In [39]:
from transformers import pipeline

pipe = pipeline("feature-extraction", model="bert-base-uncased", device=device)

In [40]:
def encode_texts(dataset):
    bs = 64
    encodings = []
    for i in tqdm(range(0, len(dataset), bs)):
        output = pipe(dataset[i:i+bs]['text'], return_tensors=True)
        encodings.extend([out[0, 0].cpu() for out in output])

    return encodings

In [41]:
dataset = load_dataset("fancyzhx/ag_news")
train_dataset = encode_texts(dataset['train'])
test_dataset = encode_texts(dataset['test'])

  0%|          | 0/1875 [00:00<?, ?it/s]

  0%|          | 0/119 [00:00<?, ?it/s]

Here, the parameter `multi_class='multinomial'` makes the regrassion multiclass

In [ ]:
from sklearn.linear_model import LogisticRegression

# multinomial logistic regression is the default for several classes (the `multi_class` argument was removed from scikit-learn)
model = LogisticRegression(max_iter=1000).fit(train_dataset, dataset['train']['label'])

predictions = model.predict(test_dataset)

In [43]:
accuracy = accuracy_metric.compute(predictions=predictions, references=dataset['test']['label'])
accuracy['accuracy']

0.9023684210526316

We obtained quite high accuracy. This means that BERT is able to extract information about the news category on its own, even though we never explicitly trained it to do so.

## BERT fine-tuning

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return accuracy.compute(predictions=np.argmax(logits, axis=-1), references=labels)


dataset = load_dataset("fancyzhx/ag_news")
split = dataset["train"].train_test_split(test_size=0.1, seed=42)
train_dataset, eval_dataset = split["train"], split["test"]
train_dataset = train_dataset.map(lambda b: tokenizer(b["text"], truncation=True, max_length=128), batched=True)
eval_dataset = eval_dataset.map(lambda b: tokenizer(b["text"], truncation=True, max_length=128), batched=True)

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./bert-agnews",
        eval_strategy="epoch",
        save_strategy="no",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=64,
        num_train_epochs=3,
        learning_rate=2e-5,
        weight_decay=0.01,
        logging_steps=50,
        report_to="none",
    ),
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.evaluate()

It’s easy to see that fine-tuning BERT directly on the classification task pushes the metrics above the baselines — just as we hoped. It would be rather disappointing to pretrain such a large language model only to lose to logistic regression, wouldn’t it?